In [20]:
# Verify the notebook is using the Python 3.11 venv (.venv311)
import sys, os
from pathlib import Path


def find_venv_python(name=".venv311", max_levels=5):
    p = Path.cwd()
    for _ in range(max_levels + 1):
        candidate = p / name
        if candidate.exists():
            venv_py = candidate / ("Scripts" if os.name == "nt" else "bin") / ("python.exe" if os.name == "nt" else "python")
            if venv_py.exists():
                return str(venv_py)
        p = p.parent
    return None

expected = find_venv_python()
print('Interpreter:', sys.executable)

if expected is None:
    # No .venv311 found in current/parent directories - give actionable advice
    print('\n\u26A0\ufe0f Could not find a local ".venv311" in the notebook folder or parent directories.')
    print('If you created the venv in the project root, make sure the kernel is registered and select it in VS Code: "Python 3.11 (.venv311)"')
    print('\nTo fix:')
    print("- In VS Code: open the kernel selector (top-right) and choose 'Python 3.11 (.venv311)'.")
    print("- Or in PowerShell run: .\\.venv311\\Scripts\\Activate.ps1 then restart the kernel.")
    print("- Or in Command Prompt run: .\\.venv311\\Scripts\\activate.bat then restart the kernel.")
    raise SystemExit("Please switch to the 'Python 3.11 (.venv311)' kernel and re-run the notebook.")

# If a candidate venv was found but the interpreter differs, show a clear message
if os.path.abspath(sys.executable) != os.path.abspath(expected):
    print('\n\u26A0\ufe0f This notebook is NOT using the Python 3.11 venv (.venv311).')
    print('Found candidate venv python:', expected)
    print('Current interpreter: ', sys.executable)
    print('\nTo fix:')
    print("- In VS Code: open the kernel selector (top-right) and choose 'Python 3.11 (.venv311)'.")
    print("- Or in PowerShell run: .\\.venv311\\Scripts\\Activate.ps1 then restart the kernel.")
    print("- Or in Command Prompt run: .\\.venv311\\Scripts\\activate.bat then restart the kernel.")
    print('\nAfter switching the kernel, re-run this cell (and then the rest of the notebook).')
    raise SystemExit("Please switch to the 'Python 3.11 (.venv311)' kernel and re-run the notebook.")

# If we reach here, the expected venv is active — verify TensorFlow can be imported
try:
    import tensorflow as tf
    print('\nTensorFlow imported successfully — version:', tf.__version__)
except Exception as e:
    print('\nFailed to import TensorFlow in the active environment:', type(e).__name__, e)
    import traceback
    traceback.print_exc()
    raise

# Check that Pillow is importable in the running kernel
try:
    import PIL
    import PIL.Image
    print('\nPillow is importable in kernel — version:', PIL.__version__)
except Exception as e:
    print('\nPillow import failed in kernel:', type(e).__name__, e)
    print('Attempting to install Pillow into the active kernel now...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'Pillow'])
    # Try import again
    try:
        import importlib
        PIL = importlib.import_module('PIL')
        importlib.reload(importlib.import_module('PIL.Image'))
        print('Pillow installed and importable — version:', PIL.__version__)
    except Exception as e2:
        print('Pillow installation succeeded but import still fails:', type(e2).__name__, e2)
        raise

# Reload potential keras utils modules so that PIL availability is re-bound
import importlib
candidates = [
    'keras.src.utils.image_utils',
    'keras.utils.image_utils',
    'tensorflow.keras.utils',
    'tensorflow.keras.utils.image_utils'
]
reloaded = False
for name in candidates:
    try:
        mod = importlib.import_module(name)
        importlib.reload(mod)
        print('Reloaded module:', name)
        reloaded = True
    except Exception as e:
        print('Could not reload', name, '-', type(e).__name__, e)

if not reloaded:
    print('\nWarning: could not find a known keras image utils module to reload. The code may continue to raise an ImportError if a stale module was imported earlier.')

Interpreter: f:\UOM\FYP\MLFlow_project\.venv311\Scripts\python.exe

TensorFlow imported successfully — version: 2.19.1

Pillow is importable in kernel — version: 12.0.0
Reloaded module: keras.src.utils.image_utils
Could not reload keras.utils.image_utils - ModuleNotFoundError No module named 'keras.utils.image_utils'
Reloaded module: tensorflow.keras.utils
Could not reload tensorflow.keras.utils.image_utils - ModuleNotFoundError No module named 'tensorflow.keras.utils.image_utils'


In [2]:
import os

os.chdir("../")
%pwd

'f:\\UOM\\FYP\\MLFlow_project'

In [3]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [4]:
from pathlib import Path
import sys, os

# Ensure we're using the project root and add the absolute `src` path to sys.path
project_root = Path.cwd()
src_path = str(project_root / "src")
print("CWD:", project_root)
print("Adding to sys.path:", src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Quick verification
import pkgutil
print("Mlflow_project present in src?", any(p.name == 'Mlflow_project' for p in pkgutil.iter_modules([src_path])))

# Now import
from Mlflow_project.constants import *
from Mlflow_project.utils.common import read_yaml, create_directories
print("Imported Mlflow_project successfully.")

CWD: f:\UOM\FYP\MLFlow_project
Adding to sys.path: f:\UOM\FYP\MLFlow_project\src
Mlflow_project present in src? True
Imported Mlflow_project successfully.


In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

        

    def get_training_config(self) -> TrainingConfig:
        training = self.config.model_trainer
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "Brain_MRI_scan_images")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [7]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [ ]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        # Load model without restored optimizer state to avoid optimizer/variable mismatches
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path,
            compile=False
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)
        
        
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        # Rebuild the model (clone) to ensure optimizer/variable sets are fresh
        try:
            weights = self.model.get_weights()
            self.model = tf.keras.models.clone_model(self.model)
            self.model.set_weights(weights)
        except Exception:
            # If cloning fails, proceed and attempt to recompile
            pass

        # Recompile the model with a fresh optimizer to avoid optimizer-variable mismatch
        try:
            from Mlflow_project.utils.common import read_yaml
            from Mlflow_project.constants import PARAMS_FILE_PATH
            params = read_yaml(PARAMS_FILE_PATH)
            lr = float(params.LEARNING_RATE)
        except Exception:
            lr = 0.01

        # Create optimizer and initialize its internal weights for current model variables
        opt = tf.keras.optimizers.SGD(learning_rate=lr)
        try:
            opt._create_all_weights(self.model.trainable_variables)
        except Exception:
            pass

        # Try standard Keras training; if optimizer/variable mismatch persists, fall back to a manual loop
        try:
            self.model.compile(
                optimizer=opt,
                loss=tf.keras.losses.CategoricalCrossentropy(),
                metrics=["accuracy"],
                run_eagerly=True
            )

            self.model.fit(
                self.train_generator,
                epochs=self.config.params_epochs,
                steps_per_epoch=self.steps_per_epoch,
                validation_steps=self.validation_steps,
                validation_data=self.valid_generator
            )

        except ValueError as e:
            # Known issue: optimizer/variable mismatch when loading a compiled model. Fall back to manual training loop.
            print('Optimizer-variable mismatch detected. Falling back to manual training loop.')
            loss_fn = tf.keras.losses.CategoricalCrossentropy()
            train_acc = tf.keras.metrics.CategoricalAccuracy()
            val_acc = tf.keras.metrics.CategoricalAccuracy()

            for epoch in range(self.config.params_epochs):
                print(f'Epoch {epoch+1}/{self.config.params_epochs}')
                train_acc.reset_states()
                steps = 0
                for _ in range(self.steps_per_epoch):
                    x_batch, y_batch = next(self.train_generator)
                    x_batch = tf.convert_to_tensor(x_batch)
                    y_batch = tf.convert_to_tensor(y_batch)
                    with tf.GradientTape() as tape:
                        preds = self.model(x_batch, training=True)
                        loss = loss_fn(y_batch, preds)
                    grads = tape.gradient(loss, self.model.trainable_variables)
                    opt.apply_gradients(zip(grads, self.model.trainable_variables))
                    train_acc.update_state(y_batch, preds)
                    steps += 1
                    if steps % 50 == 0:
                        print(f'  step {steps}/{self.steps_per_epoch} - train_acc: {train_acc.result().numpy():.4f}')

                # Validation pass (simple accuracy)
                val_acc.reset_states()
                for _ in range(self.validation_steps):
                    x_val, y_val = next(self.valid_generator)
                    preds = self.model(x_val, training=False)
                    val_acc.update_state(y_val, preds)

                print(f'  epoch {epoch+1} - train_acc: {train_acc.result().numpy():.4f}, val_acc: {val_acc.result().numpy():.4f}')

        # Save final model
        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [27]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2025-12-28 19:36:07,297: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-12-28 19:36:07,299: INFO: common: yaml file: params.yaml loaded successfully]
[2025-12-28 19:36:07,300: INFO: common: created directory at: resources]
[2025-12-28 19:36:07,301: INFO: common: created directory at: resources\model_trainer]
[2025-12-28 19:36:07,675: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 1141 images belonging to 4 classes.
Found 4571 images belonging to 4 classes.


ValueError: Unknown variable: <Variable path=dense/kernel, shape=(25088, 4), dtype=float32>. This optimizer can only be called for the variables it was originally built with. When working with a new set of variables, you should recreate a new optimizer instance.